# Module 5 — Experiment Rounds
**Day 2 | ~75 minutes**

**Goal:** Run one structured experiment round to improve the RAGAS baseline from Module 4.  
Log every experiment. Walk away with a clear evidence-based improvement backlog.

---

## How to use this notebook

Each experiment round follows the same loop:

```
1. Define config variant
2. Re-run pipeline with that config
3. Evaluate with RAGAS
4. Log result → ExperimentLog
5. Observe delta vs. baseline → discuss
```

The `ExperimentLog` persists to `data/experiment_log.json` — safe to restart kernel.

---

## Rounds

| Round | Variable | Fixed |
|-------|----------|-------|
| 1 | Chunking strategy | Retrieval, prompt |
| 2 | Retrieval method | Best chunk from Round 1, same prompt |
| 3 | Prompt template | Best from Rounds 1 + 2 |

In [ ]:
import sys, json, os
sys.path.insert(0, "..")

from openai import OpenAI
import chromadb
from dotenv import load_dotenv
from tqdm import tqdm

from src.chunker import build_chunk_records
from src.embedder import DEFAULT_MODEL
from src.vector_store import get_collection, index_chunks, retrieve
from src.retriever import hybrid_retrieve, vector_retrieve
from src.pipeline import RAGPipeline
from src.evaluator import run_evaluation
from src.experiment_log import ExperimentLog

load_dotenv("../.env")

FAST_MODEL = os.getenv("NETLIGHT_MODEL_FAST", "claude-haiku-4-5")
llm_client = OpenAI(
    base_url=os.getenv("NETLIGHT_BASE_URL", "https://llm.netlight.ai/v1"),
    api_key=os.getenv("NETLIGHT_API_KEY"),
)

# Load corpus and test questions
with open("../data/raw/corpus.json") as f:
    corpus = json.load(f)

with open("../data/sample_qa.json") as f:
    qa_data = json.load(f)

# Load cached RAG results from Module 4 (baseline)
from pathlib import Path
baseline_cache = Path("../data/rag_results.json")
with open(baseline_cache) as f:
    baseline_results = json.load(f)

# Initialise experiment log
log = ExperimentLog()
print(f"ExperimentLog loaded: {len(log)} existing entries")
print(f"Corpus: {len(corpus)} documents | Test questions: {len(qa_data['questions'])}")

---
## Helper: run pipeline + evaluate + log

This function is the core loop. We'll call it for every experiment config.

In [ ]:
def run_experiment(
    name: str,
    collection_name: str,
    chunks: list[dict],
    pipeline_kwargs: dict,
    config_meta: dict,
    notes: str = "",
    judge_model: str = FAST_MODEL,
) -> dict:
    """
    Index chunks, run pipeline on all test questions, evaluate with RAGAS, log result.

    Returns the mean RAGAS scores dict.
    """
    # 1. Get (or create) collection with this experiment's chunks
    collection = get_collection(
        collection_name=collection_name,
        persist_path="../chroma_db",
    )
    index_chunks(collection, chunks, model_name=DEFAULT_MODEL)

    # 2. Run pipeline
    rag = RAGPipeline(
        collection=collection,
        llm_client=llm_client,
        **pipeline_kwargs,
    )
    results = []
    for qa in tqdm(qa_data["questions"], desc=f"[{name}]"):
        r = rag.ask(qa["question"])
        r["ground_truth"] = qa["ground_truth"]
        results.append(r)

    # 3. RAGAS evaluation
    df = run_evaluation(results, judge_model=judge_model)
    scores = {c: float(df[c].mean()) for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"] if c in df.columns}

    # 4. Log
    log.add(name=name, config=config_meta, scores=scores, notes=notes)

    return scores


print("Helper ready. Starting experiment rounds.")

---
## Round 1 — Chunking Strategy

**Hypothesis:** Sentence-based or section-based chunking preserves more semantic context  
in structured insurance documents than fixed-size word splitting.

**Metric to watch:** Context Recall — are we retrieving chunks that contain the answer?

In [ ]:
# ── Config A: Fixed-size small (baseline already logged in Module 4) ───────
# Skip re-running if baseline is already in the log
if not any(e["name"] == "Baseline" for e in log._entries):
    baseline_scores = run_experiment(
        name="Baseline",
        collection_name="exp_baseline",
        chunks=build_chunk_records(corpus, strategy="fixed", chunk_size=200, overlap=20),
        pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
        config_meta={"strategy": "fixed", "chunk_size": 200, "overlap": 20, "top_k": 5, "retrieval": "vector"},
        notes="Day 1 baseline — fixed 200w, vector search",
    )
else:
    print("Baseline already logged — skipping.")

In [ ]:
# ── Config B: Fixed-size large ─────────────────────────────────────────────
scores_b = run_experiment(
    name="Round1-Fixed-400w",
    collection_name="exp_r1_fixed_400",
    chunks=build_chunk_records(corpus, strategy="fixed", chunk_size=400, overlap=50),
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
    config_meta={"strategy": "fixed", "chunk_size": 400, "overlap": 50, "top_k": 5, "retrieval": "vector"},
    notes="Larger chunks — less context fragmentation",
)
print(scores_b)

In [ ]:
# ── Config C: Sentence-based ───────────────────────────────────────────────
scores_c = run_experiment(
    name="Round1-Sentence-200w",
    collection_name="exp_r1_sentence",
    chunks=build_chunk_records(corpus, strategy="sentence", target_words=200, overlap_sentences=1),
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
    config_meta={"strategy": "sentence", "target_words": 200, "overlap_sentences": 1, "top_k": 5, "retrieval": "vector"},
    notes="Sentence boundaries respected — good for legal text",
)
print(scores_c)

In [ ]:
# ── Config D: Paragraph/section-based ─────────────────────────────────────
scores_d = run_experiment(
    name="Round1-Paragraph-300w",
    collection_name="exp_r1_paragraph",
    chunks=build_chunk_records(corpus, strategy="paragraph", max_words=300),
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 5},
    config_meta={"strategy": "paragraph", "max_words": 300, "top_k": 5, "retrieval": "vector"},
    notes="Paragraph/article structure preserved — best for structured insurance docs",
)
print(scores_d)

In [ ]:
# Compare Round 1 results
log.summary()
log.delta_table(baseline_name="Baseline")
log.plot()

In [ ]:
# ── Round 1 winner ─────────────────────────────────────────────────────────
# TODO: Based on results, set the best chunking config for Round 2
# Example: if paragraph chunking won context_recall, use that
BEST_CHUNK_STRATEGY = "paragraph"   # update based on your results
BEST_CHUNK_KWARGS = {"max_words": 300}

best_chunks = build_chunk_records(corpus, strategy=BEST_CHUNK_STRATEGY, **BEST_CHUNK_KWARGS)
print(f"Round 1 winner: {BEST_CHUNK_STRATEGY} | {len(best_chunks)} chunks")

---
## Round 2 — Retrieval Configuration

**Hypothesis:** Hybrid search (BM25 + vector) improves recall for technical insurance terms  
(INAMI, RIZIV, franchise, vrijstelling) that may not embed well semantically.

**Metric to watch:** Context Precision — are we retrieving *relevant* chunks (not just similar ones)?

In [ ]:
# ── Config E: Vector search top_k=3 ───────────────────────────────────────
scores_e = run_experiment(
    name="Round2-Vector-k3",
    collection_name="exp_r2_vector_k3",
    chunks=best_chunks,
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 3},
    config_meta={"strategy": BEST_CHUNK_STRATEGY, "top_k": 3, "retrieval": "vector"},
    notes="Fewer chunks — higher precision, lower recall",
)
print(scores_e)

In [ ]:
# ── Config F: Vector search top_k=8 ───────────────────────────────────────
scores_f = run_experiment(
    name="Round2-Vector-k8",
    collection_name="exp_r2_vector_k8",
    chunks=best_chunks,
    pipeline_kwargs={"llm_model": FAST_MODEL, "top_k": 8},
    config_meta={"strategy": BEST_CHUNK_STRATEGY, "top_k": 8, "retrieval": "vector"},
    notes="More context — higher recall, but potential noise",
)
print(scores_f)

In [ ]:
# ── Config G: Hybrid retrieval (BM25 + vector) ─────────────────────────────
# Note: hybrid_retrieve needs the full chunk list for BM25
# We implement this via a custom pipeline subclass

from src.retriever import hybrid_retrieve as _hybrid

class HybridRAGPipeline(RAGPipeline):
    def __init__(self, all_chunks, **kwargs):
        super().__init__(**kwargs)
        self.all_chunks = all_chunks

    def _retrieve(self, question):
        return _hybrid(
            collection=self.collection,
            all_chunks=self.all_chunks,
            question=question,
            top_k=self.top_k,
            model_name=self.embed_model_name,
        )


# Run hybrid manually (not via run_experiment helper since it uses a subclass)
collection_hybrid = get_collection("exp_r2_hybrid", persist_path="../chroma_db")
index_chunks(collection_hybrid, best_chunks, model_name=DEFAULT_MODEL)

hybrid_pipeline = HybridRAGPipeline(
    all_chunks=best_chunks,
    collection=collection_hybrid,
    llm_client=llm_client,
    llm_model=FAST_MODEL,
    top_k=5,
)

hybrid_results = []
for qa in tqdm(qa_data["questions"], desc="[Round2-Hybrid]"):
    r = hybrid_pipeline.ask(qa["question"])
    r["ground_truth"] = qa["ground_truth"]
    hybrid_results.append(r)

df_hybrid = run_evaluation(hybrid_results, judge_model=FAST_MODEL)
scores_g = {c: float(df_hybrid[c].mean()) for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"] if c in df_hybrid.columns}

log.add(
    name="Round2-Hybrid",
    config={"strategy": BEST_CHUNK_STRATEGY, "top_k": 5, "retrieval": "hybrid-BM25+vector"},
    scores=scores_g,
    notes="BM25+vector RRF fusion — good for technical insurance terms",
)
print(scores_g)

In [ ]:
# Compare Round 1 + Round 2
log.summary()
log.delta_table(baseline_name="Baseline")
log.plot()

In [ ]:
# ── Round 2 winner ─────────────────────────────────────────────────────────
BEST_RETRIEVAL = "hybrid"   # update based on results: "vector" or "hybrid"
BEST_TOP_K = 5
print(f"Round 2 winner: {BEST_RETRIEVAL}, top_k={BEST_TOP_K}")

---
## Round 3 — Prompt Templates

**Hypothesis:** Explicit language instruction and stricter grounding improve faithfulness  
while maintaining relevancy for multilingual insurance Q&A.

**Metric to watch:** Faithfulness — does the answer stick to what was retrieved?

In [ ]:
PROMPT_BASELINE = """You are a helpful insurance assistant. Answer the question using the provided context."""

PROMPT_STRICT = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided. Cite chunk numbers.
If the context is insufficient, say exactly: "I cannot answer based on the available context."
Never fabricate policy details, amounts, or conditions."""

PROMPT_MULTILINGUAL = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided. Cite chunk numbers.
IMPORTANT: Always respond in the same language as the question.
If the question is in French, answer in French. If in Dutch, answer in Dutch.
If the context is insufficient, say so in the question's language.
Never fabricate policy details, amounts, or conditions."""

PROMPT_STRUCTURED = """You are a DKV Belgium insurance assistant.
Answer ONLY using the numbered context chunks provided.
Format your answer as:
  **Answer:** [direct answer in 1-2 sentences, in the question's language]
  **Details:** [supporting detail from context, bullet points if applicable]
  **Source:** [chunk numbers used]
If the context is insufficient, state this clearly.
Never fabricate policy details, amounts, or conditions."""

prompts = {
    "Round3-Baseline-Prompt": PROMPT_BASELINE,
    "Round3-Strict-Prompt": PROMPT_STRICT,
    "Round3-Multilingual-Prompt": PROMPT_MULTILINGUAL,
    "Round3-Structured-Prompt": PROMPT_STRUCTURED,
}

print(f"Testing {len(prompts)} prompt variants")

In [ ]:
# Run all prompt variants using the best chunking + retrieval from Rounds 1+2
best_collection = get_collection("exp_r3_best", persist_path="../chroma_db")
index_chunks(best_collection, best_chunks, model_name=DEFAULT_MODEL)

for exp_name, system_prompt in prompts.items():
    rag = RAGPipeline(
        collection=best_collection,
        llm_client=llm_client,
        llm_model=FAST_MODEL,
        top_k=BEST_TOP_K,
        system_prompt=system_prompt,
    )
    results = []
    for qa in tqdm(qa_data["questions"], desc=f"[{exp_name}]"):
        r = rag.ask(qa["question"])
        r["ground_truth"] = qa["ground_truth"]
        results.append(r)

    df = run_evaluation(results, judge_model=FAST_MODEL)
    scores = {c: float(df[c].mean()) for c in ["faithfulness", "answer_relevancy", "context_precision", "context_recall"] if c in df.columns}

    log.add(
        name=exp_name,
        config={"strategy": BEST_CHUNK_STRATEGY, "top_k": BEST_TOP_K, "retrieval": BEST_RETRIEVAL, "prompt": exp_name},
        scores=scores,
        notes=f"Prompt variant: {exp_name}",
    )
    print(f"{exp_name}: {scores}")

In [ ]:
# Final comparison: all rounds
print("\n" + "="*70)
print("COMPLETE EXPERIMENT LOG")
print("="*70)
log.summary()

print("\n" + "="*70)
print("DELTA VS BASELINE")
print("="*70)
log.delta_table(baseline_name="Baseline")

log.plot()

---
## Round 4 — LLM Model Comparison

**Hypothesis:** A more capable model (sonnet) produces more faithful, relevant answers  
but at higher latency and cost. Measuring both lets you make an evidence-based call.

**Variable:** LLM model  
**Fixed:** best chunking from Round 1, best retrieval from Round 2, best prompt from Round 3

We run both models **in parallel** using `ThreadPoolExecutor` so wall-clock time  
is the slowest model, not the sum of both.


In [ ]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

SMART_MODEL = os.getenv("NETLIGHT_MODEL_SMART", "claude-sonnet-4-5-20250929")
MODELS_TO_COMPARE = {
    "haiku":  FAST_MODEL,
    "sonnet": SMART_MODEL,
}

# TODO:
# 1. Define run_pipeline_for_model(label, model_id) that:
#    - Creates a RAGPipeline with that model, BEST_TOP_K, and best_collection
#    - Loops over qa_data["questions"], calling rag.ask() and measuring time.perf_counter() latency
#    - Returns {"label", "results": [...], "latencies": [...]}
#
# 2. Use ThreadPoolExecutor(max_workers=len(MODELS_TO_COMPARE)) to run all models concurrently.
#    Collect results with as_completed(futures).
#
# 3. For each model's results, call run_evaluation(results, llm_client=llm_client, judge_model=FAST_MODEL).
#
# 4. Log each to ExperimentLog with name="Round4-{label}".
#
# 5. Print a side-by-side score summary and a latency vs faithfulness scatter plot.


---
## Multilingual Spot-Check

Run the best pipeline on specific FR/NL/EN test questions and manually verify language fidelity.

In [ ]:
# Find best prompt variant from log
df_log = log.to_dataframe()
r3_entries = df_log[df_log["name"].str.startswith("Round3")]
best_prompt_name = r3_entries.loc[r3_entries["faithfulness"].idxmax(), "name"] if len(r3_entries) else "Round3-Multilingual-Prompt"
best_prompt = prompts.get(best_prompt_name, PROMPT_MULTILINGUAL)

print(f"Best prompt: {best_prompt_name}")

best_rag = RAGPipeline(
    collection=best_collection,
    llm_client=llm_client,
    llm_model=FAST_MODEL,
    top_k=BEST_TOP_K,
    system_prompt=best_prompt,
)

multilingual_questions = [
    ("FR", "Quelle est la franchise annuelle pour l'hospitalisation?"),
    ("NL", "Wat is het maximale dagbedrag voor kinesitherapie bij een geconventioneerde kinesitherapeut?"),
    ("EN", "What is the maximum coverage per day for a single room?"),
    ("FR", "Quels documents faut-il fournir pour un remboursement d'hospitalisation?"),
    ("NL", "Worden tandheelkundige kosten vergoed na een verkeersongeval?"),
]

print("\n" + "="*70)
for lang, question in multilingual_questions:
    result = best_rag.ask(question)
    print(f"\n[{lang}] Q: {question}")
    print(f"    A: {result['answer'][:300]}")
    print(f"    Sources: {list(set(result['sources']))}")

---
## Improvement Backlog

Use the experiment results to populate the backlog for post-workshop work.

In [ ]:
# Print experiment summary for the backlog discussion
df_log = log.to_dataframe()

# Find best config per metric
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
print("Best configuration per metric:")
for m in metrics:
    if m in df_log.columns:
        best_idx = df_log[m].idxmax()
        best_row = df_log.loc[best_idx]
        print(f"  {m:<25}: {best_row['name']} ({best_row[m]:.3f})")

print("\n\nIMPROVEMENT BACKLOG TEMPLATE")
print("="*60)
print("""
Priority 1 (High impact, measured):
  [ ] TODO: fill based on experiment results

Priority 2 (Medium impact, measured):  
  [ ] TODO: fill based on experiment results

Priority 3 (Untested, next sprint):
  [ ] Re-ranking with cross-encoder (add to hybrid pipeline)
  [ ] Fine-tune multilingual embedding model on DKV terminology
  [ ] Add query expansion for FR/NL synonyms (INAMI vs RIZIV vs NIHDI)
  [ ] Evaluate with native-speaker review of FR/NL answers
  [ ] Production deployment: FastAPI wrapper + Docker
""")

In [ ]:
# Save the full experiment log as CSV for sharing
csv_path = "../data/experiment_log_export.csv"
df_log.to_csv(csv_path, index=False)
print(f"Experiment log exported to {csv_path}")
print("\nEnd of workshop. The team now owns:")
print("  ✓ Working RAG pipeline on DKV Belgium documents")
print("  ✓ RAGAS evaluation harness")
print("  ✓ Complete experiment log with deltas")
print("  ✓ Evidence-based improvement backlog")